# Regex Baseline Evaluation

This notebook evaluates the regex-based PII detector against the synthetic dataset, establishing the baseline that the Week 2 BERT model will be compared against.

In [1]:
import sys
import json
sys.path.insert(0, '../src')

from data.generator import Entity, generate_dataset
from baseline.regex_detector import detect_pii
from evaluation.metrics import evaluate

## Load the dataset

In [2]:
with open('../data/processed/synthetic_v1.jsonl') as f:
    items = [json.loads(line) for line in f]

print(f"Loaded {len(items)} examples")

Loaded 2000 examples


## Run the regex detector and collect predictions

In [3]:
true_per_example = []
pred_per_example = []

for item in items:
    text = item["text"]
    true_entities = [Entity(**e) for e in item["entities"]]
    pred_entities = detect_pii(text)
    true_per_example.append(true_entities)
    pred_per_example.append(pred_entities)

## Results: precision / recall / F1 by category

In [4]:
results = evaluate(true_per_example, pred_per_example)

categories = ["EMAIL", "PHONE", "SSN", "CREDIT_CARD", "PERSON_NAME", "ADDRESS", "OVERALL"]

print(f'{"Category":15} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print("-" * 47)
for cat in categories:
    if cat in results:
        m = results[cat]
        print(f"{cat:15} {m.precision:>10.3f} {m.recall:>10.3f} {m.f1:>10.3f}")

Category         Precision     Recall         F1
-----------------------------------------------
EMAIL                1.000      1.000      1.000
PHONE                1.000      1.000      1.000
SSN                  1.000      1.000      1.000
CREDIT_CARD          1.000      1.000      1.000
PERSON_NAME          0.000      0.000      0.000
ADDRESS              0.000      0.000      0.000
OVERALL              1.000      0.569      0.725


## Example predictions

A few examples showing what the detector catches and what it misses.

In [5]:
for item in items[:8]:
    text = item["text"]
    true_types = {e["type"] for e in item["entities"]}
    pred_types = {e.type for e in detect_pii(text)}
    missed = true_types - pred_types
    flag = " <-- MISSED: " + ", ".join(missed) if missed else ""
    print(f"{text[:65]:65}{flag}")

Contact Rachel Robinson at sandysnyder@example.org.               <-- MISSED: PERSON_NAME
Reach out to Patricia Moore via reevesdouglas@example.net or (369 <-- MISSED: PERSON_NAME
You can call Phillip Salazar on (499) 939-3305 anytime.           <-- MISSED: PERSON_NAME
The weather today is quite pleasant.                             
Ship the package to 46657 Christy Haven Apt. 722, East Catherine, <-- MISSED: ADDRESS
You can call Mrs. Veronica Gould MD on 658-202-9702 anytime.      <-- MISSED: PERSON_NAME
Reach out to Corey Sandoval via angelagrant@example.net or +1-195 <-- MISSED: PERSON_NAME
Let me know if you have any questions.                           


## Takeaway

The regex baseline achieves perfect precision and recall on structured PII (EMAIL, PHONE, SSN, CREDIT_CARD), since these follow rigid, learnable formats. It scores 0 on PERSON_NAME and ADDRESS, since recognizing these requires contextual language understanding that pattern matching cannot provide.

This is the gap Week 2's fine-tuned BERT model is designed to close.